Question 1: Incremental Upsert (Python)
Your issues:

Wrong imports: from spark.sql → should be from pyspark.sql (but this was pure Python, no Spark needed)
Used inner join — loses new records (id=3 dropped!)
Missing closing parenthesis
id not quoted — should be "id" or col("id")
They wanted plain Python, not PySpark
Correct answer (what they wanted):

def merge_batch(existing_table: list[dict], incoming_batch: list[dict]) -> list[dict]:
    """
    Idempotent upsert: merge incoming into existing.
    - id = primary key
    - Keep most recent record per id (updated_at)
    - Running twice with same input = same result (idempotent)
    """
    # Combine all records
    combined = {r["id"]: r for r in existing_table}
    
    for record in incoming_batch:
        rid = record["id"]
        if rid not in combined or record["updated_at"] > combined[rid]["updated_at"]:
            combined[rid] = record
    
    return sorted(combined.values(), key=lambda x: x["id"])


# Test
existing = [
    {"id": 1, "value": 10, "updated_at": "2024-01-01T10:00:00"},
    {"id": 2, "value": 20, "updated_at": "2024-01-02T10:00:00"},
]
incoming = [
    {"id": 2, "value": 25, "updated_at": "2024-01-03T09:00:00"},
    {"id": 3, "value": 30, "updated_at": "2024-01-03T09:05:00"},
]

result = merge_batch(existing, incoming)
# Result:
# id=1: value=10 (unchanged, no incoming)
# id=2: value=25 (updated, incoming is newer)
# id=3: value=30 (new record, inserted)
Why idempotent: Run same function 10 times with same input → same output. Dict keyed by id = natural dedup.

Key lesson: When they say "Python function" with list[dict] input — they want pure Python, not PySpark. Read input types carefully.

Question 2: Rolling Window SQL
Your issues:

No DAU count per day (needed COUNT(DISTINCT user_id))
Used LAG instead of AVG() OVER(ROWS BETWEEN)
WHERE with interval wrong — doesn't give rolling average
Missing rolling 3-day window logic entirely
Correct answer:

WITH daily_active AS (
    SELECT 
        event_date,
        COUNT(DISTINCT user_id) AS dau
    FROM events
    GROUP BY event_date
)
SELECT 
    event_date,
    dau,
    ROUND(
        AVG(dau) OVER (
            ORDER BY event_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2
    ) AS rolling_3day_avg
FROM daily_active
ORDER BY event_date;
Key concepts:

What	How
DAU per day	COUNT(DISTINCT user_id) GROUP BY event_date
Rolling 3-day avg	AVG() OVER(ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
Why CTE first	Need to aggregate DAU before applying window
Key lesson: ROWS BETWEEN N PRECEDING AND CURRENT ROW = rolling window. This is different from LAG (which gets single previous value).

Question 3: Airflow DAG
Your answer was directionally correct. Structure was right: ingest → transform → load → notify. Good instinct on run_spark_job() and hooks.

Issues:

Missing SLA definition
Missing how to avoid partial writes
Syntax errors in dag decorator
check_row_count not wired in
Correct answer:

from datetime import datetime, timedelta
from airflow.decorators import dag, task
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
from airflow.providers.snowflake.hooks.snowflake import SnowflakeHook
from airflow.operators.email import EmailOperator
from airflow.utils.trigger_rule import TriggerRule
import subprocess
import json

DEFAULT_ARGS = {
    "owner": "data-engineering",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "email_on_failure": True,
    "email": ["team@company.com"],
    "sla": timedelta(hours=2),  # SLA: must finish within 2 hours
}

def run_spark_job(script: str, args: list = None) -> dict:
    cmd = [
        "spark-submit",
        "--master", "yarn",
        "--deploy-mode", "client",
        script,
    ] + (args or [])

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)

    if proc.returncode != 0:
        raise RuntimeError(f"Spark failed: {proc.stderr[-1000:]}")

    # Parse JSON metadata from last stdout line
    for line in reversed(proc.stdout.strip().split("\n")):
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    return {"status": "success"}


@dag(
    dag_id="csv_to_snowflake_daily",
    default_args=DEFAULT_ARGS,
    schedule="0 6 * * *",
    start_date=datetime(2025, 1, 1),
    catchup=False,
    max_active_runs=1,
)
def csv_pipeline():

    # 1. Wait for CSV in S3
    wait_csv = S3KeySensor(
        task_id="wait_csv_file",
        bucket_name="raw-bucket",
        bucket_key="daily/{{ ds }}/data.csv",
        aws_conn_id="aws_default",
        poke_interval=300,
        timeout=7200,
        mode="reschedule",
    )

    # 2. Validate source
    @task()
    def validate_source(ds=None) -> dict:
        from airflow.providers.amazon.aws.hooks.s3 import S3Hook
        hook = S3Hook(aws_conn_id="aws_default")
        metadata = hook.head_object(
            key=f"daily/{ds}/data.csv",
            bucket_name="raw-bucket"
        )
        size_mb = metadata["ContentLength"] / (1024 * 1024)
        if size_mb < 0.001:
            raise ValueError(f"File too small: {size_mb}MB")
        return {"path": f"s3://raw-bucket/daily/{ds}/data.csv", "size_mb": size_mb}

    # 3. Transform (Spark)
    @task()
    def transform(source_info: dict) -> dict:
        result = run_spark_job(
            "/opt/spark/transform_csv.py",
            args=["--source", source_info["path"]]
        )
        return {"silver_path": result.get("output_path", ""), "count": result.get("record_count", 0)}

    # 4. Load to Snowflake — AVOIDS PARTIAL WRITES
    @task()
    def load_snowflake(transform_info: dict) -> dict:
        """
        Avoid partial writes strategy:
        1. Write to staging table first
        2. MERGE (upsert) from staging to target in single transaction
        3. If anything fails, staging is disposable
        """
        hook = SnowflakeHook(snowflake_conn_id="snowflake_default")
        
        # Atomic: COPY INTO staging, then MERGE into target
        hook.run(f"""
            COPY INTO staging_table
            FROM @s3_stage/{transform_info['silver_path']}
            FILE_FORMAT = (TYPE = 'PARQUET');
        """)
        hook.run(f"""
            MERGE INTO target_table t
            USING staging_table s ON t.id = s.id
            WHEN MATCHED THEN UPDATE SET t.value = s.value
            WHEN NOT MATCHED THEN INSERT (id, value) VALUES (s.id, s.value);
        """)
        hook.run("TRUNCATE TABLE staging_table;")
        
        return {"status": "loaded", "records": transform_info["count"]}

    # 5. Quality check
    @task()
    def quality_check(load_info: dict) -> dict:
        hook = SnowflakeHook(snowflake_conn_id="snowflake_default")
        count = hook.get_first("SELECT COUNT(*) FROM target_table")[0]
        if count == 0:
            raise ValueError("Zero records in target!")
        return {"status": "passed", "count": count}

    # 6. Notifications
    notify_success = EmailOperator(
        task_id="notify_success",
        to=["team@company.com"],
        subject="Pipeline {{ ds }} SUCCESS",
        html_content="<p>Loaded {{ ti.xcom_pull(task_ids='quality_check')['count'] }} records</p>",
        trigger_rule=TriggerRule.ALL_SUCCESS,
    )

    notify_failure = EmailOperator(
        task_id="notify_failure",
        to=["team@company.com"],
        subject="Pipeline {{ ds }} FAILED",
        html_content="<p>Check Airflow logs</p>",
        trigger_rule=TriggerRule.ONE_FAILED,
    )

    # Wiring
    source = validate_source()
    wait_csv >> source
    silver = transform(source)
    loaded = load_snowflake(silver)
    qc = quality_check(loaded)
    qc >> notify_success
    qc >> notify_failure

csv_pipeline()
Partial writes answer: "I use staging table + MERGE pattern. Data loads into disposable staging first, then single atomic MERGE into target. If pipeline fails mid-way, target table untouched. Staging gets truncated next run."